In [0]:
%pip install -U kaleido
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")


In [0]:
import pandas as pd
from src.data.collectors.weather import _DNO_REGIONS

# Load the summary df from both models
results_04 = pd.read_csv("/Volumes/workspace/default/models/04_train_regional_models.csv")
results_04 = results_04[results_04['region_id'] <= 14]

results_08 = pd.read_csv("/Volumes/workspace/default/models/08_train_regional_models_with_weather.csv")


print(f"Model 04 (without weather): {len(results_04)} rows")
print(f"Model 08 (with weather): {len(results_08)} rows")

# Display both
print("\n=== Model without weather ===")
print(results_04.head())

print("\n=== Model with weather ===")
print(results_08.head())

In [0]:
# Add model type to both
results_04['model_type'] = 'no_weather'
results_08['model_type'] = 'with_weather'


# Filter by alpha
results_04_50 = results_04[results_04['alpha'] == 0.5]
results_08_50 = results_08[results_08['alpha'] == 0.5]

# Combine them for comparison
alpha_50_combined = pd.concat([results_04_50, results_08_50])


%pip install plotly
import plotly.express as px

x = alpha_50_combined["region"]
y = alpha_50_combined["pinball_loss"]
color = alpha_50_combined["model_type"]
barmode = "group"

# Plot the results
fig = px.bar(x=x, y=y, color=color, barmode=barmode, labels={"color": "Model Type"})
fig.show()

In [ ]:
import plotly.express as px

# Add model type labels and filter to P50
results_04['model_type'] = 'no_weather'
results_08['model_type'] = 'with_weather'

results_04_50 = results_04[results_04['alpha'] == 0.5]
results_08_50 = results_08[results_08['alpha'] == 0.5]

alpha_50_combined = pd.concat([results_04_50, results_08_50])

# Grouped bar chart: baseline vs weather models by region
fig = px.bar(
    alpha_50_combined,
    x="region",
    y="pinball_loss",
    color="model_type",
    barmode="group",
    labels={"region": "DNO Region", "pinball_loss": "P50 Pinball Loss", "model_type": "Model Type"},
    title="P50 Pinball Loss by Region: Baseline vs Weather Features",
    color_discrete_map={"no_weather": "#636EFA", "with_weather": "#EF553B"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.write_image(f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/docs/images/comapre_models.png")
fig.show()

In [ ]:
# Geoplot: P50 pinball loss mapped across all 14 DNO regions
fig = px.scatter_mapbox(
    results_08_50_combined,
    lat="latitude",
    lon="longitude",
    color="pinball_loss",
    size="pinball_loss",
    hover_name="region",
    hover_data={"latitude": False, "longitude": False, "Accuracy": True, "pinball_loss": True},
    color_continuous_scale="Viridis",
    mapbox_style="open-street-map",
    zoom=4,
    center={"lat": 54.5, "lon": -2.5},
    title="P50 Pinball Loss by DNO Region"
)
fig.write_image(f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/docs/images/Models_Geoplot.png")
fig.show()